# CUDA Kernel Example: Self-Attention

This notebook implements the **Scaled Dot-Product Self-Attention** mechanism using custom CUDA kernels via **PyCUDA**.

## Self-Attention Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Steps:
1. **Linear projections** — compute Q, K, V from input X
2. **Attention scores** — compute `QKᵀ / sqrt(d_k)`
3. **Softmax** — row-wise softmax over scores
4. **Output** — multiply softmax weights by V

### CUDA Kernels implemented:
| Kernel | Purpose |
|---|---|
| `matmul_kernel` | Tiled matrix multiplication (used for QKᵀ and Attn·V) |
| `scale_kernel` | Element-wise scale by `1/sqrt(d_k)` |
| `softmax_kernel` | Row-wise softmax (numerically stable) |
| `linear_proj_kernel` | Linear projection X·W for Q, K, V |

## 1. Install & Import Dependencies

In [13]:
%pip install pycuda numpy

/home/per/nbi/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import numpy as np
import pycuda.autoinit                  # automatically initialise CUDA device
import pycuda.driver as cuda
from pycuda.compiler import SourceModule

print(f"PyCUDA ready  —  device: {cuda.Device(0).name()}")

PyCUDA ready  —  device: NVIDIA GeForce RTX 4070 SUPER


## 2. CUDA Kernel Source Code

In [15]:
CUDA_SOURCE = r"""
// ─────────────────────────────────────────────────────────────────────────────
// Tile size for shared-memory tiled matrix multiplication
// ─────────────────────────────────────────────────────────────────────────────
#define TILE 32

// ─────────────────────────────────────────────────────────────────────────────
// Kernel 1 – Tiled Matrix Multiplication
//   C  [M x N]  =  A [M x K]  *  B [K x N]   (row-major)
// ─────────────────────────────────────────────────────────────────────────────
__global__ void matmul_kernel(
        const float* __restrict__ A,
        const float* __restrict__ B,
        float*       __restrict__ C,
        int M, int K, int N)
{
    __shared__ float sA[TILE][TILE];
    __shared__ float sB[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    float acc = 0.0f;

    for (int t = 0; t < (K + TILE - 1) / TILE; ++t) {
        int aCol = t * TILE + threadIdx.x;
        sA[threadIdx.y][threadIdx.x] = (row < M && aCol < K) ? A[row * K + aCol] : 0.0f;

        int bRow = t * TILE + threadIdx.y;
        sB[threadIdx.y][threadIdx.x] = (bRow < K && col < N) ? B[bRow * N + col] : 0.0f;

        __syncthreads();

        #pragma unroll
        for (int i = 0; i < TILE; ++i)
            acc += sA[threadIdx.y][i] * sB[i][threadIdx.x];

        __syncthreads();
    }

    if (row < M && col < N)
        C[row * N + col] = acc;
}

// ─────────────────────────────────────────────────────────────────────────────
// Kernel 2 – Element-wise Scale
//   out[i] = in[i] * scale
// ─────────────────────────────────────────────────────────────────────────────
__global__ void scale_kernel(float* data, float scale, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n)
        data[idx] *= scale;
}

// ─────────────────────────────────────────────────────────────────────────────
// Kernel 3 – Row-wise Numerically-Stable Softmax
// ─────────────────────────────────────────────────────────────────────────────
__global__ void softmax_kernel(float* scores, int seq_len)
{
    extern __shared__ float smem[];

    int row = blockIdx.x;
    int tid = threadIdx.x;

    float* row_ptr = scores + row * seq_len;

    float thread_max = -1e38f;
    for (int i = tid; i < seq_len; i += blockDim.x)
        thread_max = fmaxf(thread_max, row_ptr[i]);

    smem[tid] = thread_max;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride)
            smem[tid] = fmaxf(smem[tid], smem[tid + stride]);
        __syncthreads();
    }
    float row_max = smem[0];

    float* exp_cache = smem + blockDim.x;

    float thread_sum = 0.0f;
    for (int i = tid; i < seq_len; i += blockDim.x) {
        float e = expf(row_ptr[i] - row_max);
        exp_cache[i] = e;
        thread_sum += e;
    }

    smem[tid] = thread_sum;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride)
            smem[tid] += smem[tid + stride];
        __syncthreads();
    }
    float row_sum = smem[0];

    for (int i = tid; i < seq_len; i += blockDim.x)
        row_ptr[i] = exp_cache[i] / row_sum;
}

// ─────────────────────────────────────────────────────────────────────────────
// Kernel 4 – Linear Projection   out = X · W
// ─────────────────────────────────────────────────────────────────────────────
__global__ void linear_proj_kernel(
        const float* __restrict__ X,
        const float* __restrict__ W,
        float*       __restrict__ out,
        int seq_len, int d_in, int d_out)
{
    __shared__ float sX[TILE][TILE];
    __shared__ float sW[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    float acc = 0.0f;

    for (int t = 0; t < (d_in + TILE - 1) / TILE; ++t) {
        int xCol = t * TILE + threadIdx.x;
        sX[threadIdx.y][threadIdx.x] = (row < seq_len && xCol < d_in) ? X[row * d_in + xCol] : 0.0f;

        int wRow = t * TILE + threadIdx.y;
        sW[threadIdx.y][threadIdx.x] = (wRow < d_in && col < d_out) ? W[wRow * d_out + col] : 0.0f;

        __syncthreads();

        #pragma unroll
        for (int i = 0; i < TILE; ++i)
            acc += sX[threadIdx.y][i] * sW[i][threadIdx.x];

        __syncthreads();
    }

    if (row < seq_len && col < d_out)
        out[row * d_out + col] = acc;
}

// ─────────────────────────────────────────────────────────────────────────────
// Kernel 5 – Coalesced In-Place Transpose
//   dst [cols x rows]  =  src [rows x cols]
//   Padding (+1) on shared memory tile eliminates bank conflicts.
// ─────────────────────────────────────────────────────────────────────────────
__global__ void transpose_kernel(
        const float* __restrict__ src,
        float*       __restrict__ dst,
        int rows, int cols)
{
    __shared__ float tile[TILE][TILE + 1];   // +1 avoids shared-memory bank conflicts

    int src_x = blockIdx.x * TILE + threadIdx.x;   // col in src
    int src_y = blockIdx.y * TILE + threadIdx.y;   // row in src

    if (src_y < rows && src_x < cols)
        tile[threadIdx.y][threadIdx.x] = src[src_y * cols + src_x];

    __syncthreads();

    // Write out transposed: swap block indices
    int dst_x = blockIdx.y * TILE + threadIdx.x;   // col in dst  (= row in src)
    int dst_y = blockIdx.x * TILE + threadIdx.y;   // row in dst  (= col in src)

    if (dst_y < cols && dst_x < rows)
        dst[dst_y * rows + dst_x] = tile[threadIdx.x][threadIdx.y];
}
"""

mod = SourceModule(CUDA_SOURCE)

# Retrieve compiled kernel handles
matmul_fn    = mod.get_function("matmul_kernel")
scale_fn     = mod.get_function("scale_kernel")
softmax_fn   = mod.get_function("softmax_kernel")
linproj_fn   = mod.get_function("linear_proj_kernel")
transpose_fn = mod.get_function("transpose_kernel")

print("All CUDA kernels compiled successfully ✓")

All CUDA kernels compiled successfully ✓


## 3. Helper Utilities

In [16]:
import math

TILE = 32   # must match #define TILE in CUDA source

def ceil_div(a, b):
    return (a + b - 1) // b

def gpu_alloc_like(arr):
    """Allocate GPU memory and copy a numpy array to it."""
    gpu_buf = cuda.mem_alloc(arr.nbytes)
    cuda.memcpy_htod(gpu_buf, arr)
    return gpu_buf

def gpu_zeros(shape, dtype=np.float32):
    """Allocate zeroed GPU buffer."""
    arr = np.zeros(shape, dtype=dtype)
    gpu_buf = cuda.mem_alloc(arr.nbytes)
    cuda.memcpy_htod(gpu_buf, arr)
    return gpu_buf, arr

def from_gpu(gpu_buf, shape, dtype=np.float32):
    """Copy GPU buffer back to a numpy array."""
    arr = np.empty(shape, dtype=dtype)
    cuda.memcpy_dtoh(arr, gpu_buf)
    return arr

def matmul_gpu(A_gpu, B_gpu, M, K, N):
    """Launch matmul_kernel: C = A @ B  (sizes M×K, K×N → M×N)."""
    C_gpu, _ = gpu_zeros((M, N))
    grid  = (ceil_div(N, TILE), ceil_div(M, TILE), 1)
    block = (TILE, TILE, 1)
    matmul_fn(A_gpu, B_gpu, C_gpu,
              np.int32(M), np.int32(K), np.int32(N),
              block=block, grid=grid)
    return C_gpu

def transpose_gpu(src_gpu, rows, cols):
    """Launch transpose_kernel: dst [cols x rows] = src [rows x cols]  (GPU only)."""
    dst_gpu, _ = gpu_zeros((cols, rows))
    grid  = (ceil_div(cols, TILE), ceil_div(rows, TILE), 1)
    block = (TILE, TILE, 1)
    transpose_fn(src_gpu, dst_gpu, np.int32(rows), np.int32(cols),
                 block=block, grid=grid)
    return dst_gpu

print("Helper utilities defined ✓")

Helper utilities defined ✓


## 4. Self-Attention Function

In [17]:
def cuda_self_attention(X: np.ndarray,
                        Wq: np.ndarray,
                        Wk: np.ndarray,
                        Wv: np.ndarray) -> np.ndarray:
    """
    Scaled dot-product self-attention computed entirely on the GPU.

    Parameters
    ----------
    X  : (seq_len, d_model)  — input sequence
    Wq : (d_model, d_k)      — query projection weights
    Wk : (d_model, d_k)      — key projection weights
    Wv : (d_model, d_v)      — value projection weights

    Returns
    -------
    out : (seq_len, d_v)  — attention output
    """
    X  = X.astype(np.float32)
    Wq = Wq.astype(np.float32)
    Wk = Wk.astype(np.float32)
    Wv = Wv.astype(np.float32)

    seq_len, d_model = X.shape
    _, d_k = Wq.shape
    _, d_v = Wv.shape

    # ── Upload inputs to GPU ─────────────────────────────────────────────────
    X_gpu  = gpu_alloc_like(X)
    Wq_gpu = gpu_alloc_like(Wq)
    Wk_gpu = gpu_alloc_like(Wk)
    Wv_gpu = gpu_alloc_like(Wv)

    # ── Step 1 : Linear projections  Q = X·Wq,  K = X·Wk,  V = X·Wv ────────
    def proj(X_gpu, W_gpu, d_out):
        out_gpu, _ = gpu_zeros((seq_len, d_out))
        grid  = (ceil_div(d_out, TILE), ceil_div(seq_len, TILE), 1)
        block = (TILE, TILE, 1)
        linproj_fn(X_gpu, W_gpu, out_gpu,
                   np.int32(seq_len), np.int32(d_model), np.int32(d_out),
                   block=block, grid=grid)
        return out_gpu

    Q_gpu = proj(X_gpu, Wq_gpu, d_k)   # (seq_len, d_k)
    K_gpu = proj(X_gpu, Wk_gpu, d_k)   # (seq_len, d_k)
    V_gpu = proj(X_gpu, Wv_gpu, d_v)   # (seq_len, d_v)

    print(f"  ✓ Q, K, V projections  [{seq_len} × {d_k}] each (V: [{seq_len} × {d_v}])")

    # ── Step 2 : Transpose K on GPU  Kᵀ [d_k × seq_len] ─────────────────────
    #    No CPU round-trip: transpose_kernel handles this entirely on device.
    Kt_gpu = transpose_gpu(K_gpu, rows=seq_len, cols=d_k)   # (d_k, seq_len)

    # ── Step 3 : Attention scores  S = Q · Kᵀ ────────────────────────────────
    S_gpu = matmul_gpu(Q_gpu, Kt_gpu, seq_len, d_k, seq_len)   # (seq_len, seq_len)
    print(f"  ✓ Raw scores  S = Q·Kᵀ  [{seq_len} × {seq_len}]")

    # ── Step 4 : Scale by 1 / sqrt(d_k) ─────────────────────────────────────
    scale = np.float32(1.0 / math.sqrt(d_k))
    n_elements = seq_len * seq_len
    threads = 256
    scale_fn(S_gpu, scale, np.int32(n_elements),
             block=(threads, 1, 1),
             grid=(ceil_div(n_elements, threads), 1, 1))
    print(f"  ✓ Scaled scores  (÷ √{d_k} = {scale:.4f})")

    # ── Step 5 : Row-wise softmax ─────────────────────────────────────────────
    threads_sm   = min(128, seq_len)
    shared_bytes = (threads_sm + seq_len) * 4
    softmax_fn(S_gpu, np.int32(seq_len),
               block=(threads_sm, 1, 1),
               grid=(seq_len, 1, 1),
               shared=shared_bytes)
    print(f"  ✓ Softmax attention weights  [{seq_len} × {seq_len}]")

    # ── Step 6 : Output  Out = Attn · V ──────────────────────────────────────
    Out_gpu = matmul_gpu(S_gpu, V_gpu, seq_len, seq_len, d_v)   # (seq_len, d_v)
    print(f"  ✓ Output  Out = Attn·V  [{seq_len} × {d_v}]")

    # ── Download result ───────────────────────────────────────────────────────
    out = from_gpu(Out_gpu, (seq_len, d_v))

    # Free GPU memory explicitly
    for buf in [X_gpu, Wq_gpu, Wk_gpu, Wv_gpu,
                Q_gpu, K_gpu, V_gpu, Kt_gpu, S_gpu, Out_gpu]:
        buf.free()

    return out

print("cuda_self_attention() defined ✓")

cuda_self_attention() defined ✓


## 5. Run & Verify Against NumPy Reference

In [23]:
def numpy_self_attention(X, Wq, Wk, Wv):
    """Pure-NumPy reference implementation."""
    Q = X @ Wq                             # (seq, d_k)
    K = X @ Wk
    V = X @ Wv                             # (seq, d_v)
    d_k = Q.shape[-1]
    S   = Q @ K.T / math.sqrt(d_k)        # (seq, seq)
    S   = S - S.max(axis=1, keepdims=True) # numerical stability
    A   = np.exp(S)
    A  /= A.sum(axis=1, keepdims=True)    # softmax
    return A @ V                           # (seq, d_v)

# ── Hyper-parameters ─────────────────────────────────────────────────────────
SEQ_LEN = 64     # number of tokens
D_MODEL = 128    # input embedding dimension
D_K     = 64     # query / key dimension
D_V     = 64     # value / output dimension

rng = np.random.default_rng(42)
X  = rng.standard_normal((SEQ_LEN, D_MODEL)).astype(np.float32)
Wq = rng.standard_normal((D_MODEL, D_K)).astype(np.float32) * 0.02
Wk = rng.standard_normal((D_MODEL, D_K)).astype(np.float32) * 0.02
Wv = rng.standard_normal((D_MODEL, D_V)).astype(np.float32) * 0.02

print("=" * 55)
print(f"  Self-Attention  |  seq={SEQ_LEN}  d_model={D_MODEL}  d_k={D_K}  d_v={D_V}")
print("=" * 55)
print("\n── CUDA execution ──────────────────────────────────────")
cuda_out = cuda_self_attention(X, Wq, Wk, Wv)

print("\n── NumPy reference ─────────────────────────────────────")
ref_out  = numpy_self_attention(X, Wq, Wk, Wv)

# ── Numerical comparison ──────────────────────────────────────────────────────
max_err = np.max(np.abs(cuda_out - ref_out))
mean_err = np.mean(np.abs(cuda_out - ref_out))
print(f"\n── Comparison ──────────────────────────────────────────")
print(f"  Output shape  : {cuda_out.shape}")
print(f"  Max  |error|  : {max_err:.2e}")
print(f"  Mean |error|  : {mean_err:.2e}")
print(f"  {'✓ PASS — results match reference!' if max_err < 1e-4 else '✗ FAIL — large numerical discrepancy'}")

  Self-Attention  |  seq=64  d_model=128  d_k=64  d_v=64

── CUDA execution ──────────────────────────────────────
  ✓ Q, K, V projections  [64 × 64] each (V: [64 × 64])
  ✓ Raw scores  S = Q·Kᵀ  [64 × 64]
  ✓ Scaled scores  (÷ √64 = 0.1250)
  ✓ Softmax attention weights  [64 × 64]
  ✓ Output  Out = Attn·V  [64 × 64]

── NumPy reference ─────────────────────────────────────

── Comparison ──────────────────────────────────────────
  Output shape  : (64, 64)
  Max  |error|  : 2.24e-08
  Mean |error|  : 2.80e-09
  ✓ PASS — results match reference!


## 6. Performance Benchmark

In [19]:
import time

def benchmark(fn, *args, warmup=3, runs=10):
    for _ in range(warmup):
        fn(*args)
    cuda.Context.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        fn(*args)
    cuda.Context.synchronize()
    return (time.perf_counter() - t0) / runs * 1e3   # ms per call

configs = [
    (32,  64,  32,  32),
    (64,  128, 64,  64),
    (128, 256, 128, 128),
    (256, 512, 256, 256),
]

print(f"{'seq':>5} {'d_model':>8} {'d_k':>5} {'CUDA (ms)':>12} {'NumPy (ms)':>12} {'Speedup':>9}")
print("-" * 56)

for seq, dm, dk, dv in configs:
    rng2 = np.random.default_rng(0)
    X2   = rng2.standard_normal((seq, dm)).astype(np.float32)
    Wq2  = rng2.standard_normal((dm, dk)).astype(np.float32) * 0.02
    Wk2  = rng2.standard_normal((dm, dk)).astype(np.float32) * 0.02
    Wv2  = rng2.standard_normal((dm, dv)).astype(np.float32) * 0.02

    t_cuda  = benchmark(cuda_self_attention, X2, Wq2, Wk2, Wv2)
    t_numpy = benchmark(numpy_self_attention, X2, Wq2, Wk2, Wv2)
    speedup = t_numpy / t_cuda

    print(f"{seq:>5} {dm:>8} {dk:>5} {t_cuda:>11.3f}  {t_numpy:>11.3f}  {speedup:>8.2f}×")

  seq  d_model   d_k    CUDA (ms)   NumPy (ms)   Speedup
--------------------------------------------------------
  ✓ Q, K, V projections  [32 × 32] each (V: [32 × 32])
  ✓ Raw scores  S = Q·Kᵀ  [32 × 32]
  ✓ Scaled scores  (÷ √32 = 0.1768)
  ✓ Softmax attention weights  [32 × 32]
  ✓ Output  Out = Attn·V  [32 × 32]
  ✓ Q, K, V projections  [32 × 32] each (V: [32 × 32])
  ✓ Raw scores  S = Q·Kᵀ  [32 × 32]
  ✓ Scaled scores  (÷ √32 = 0.1768)
  ✓ Softmax attention weights  [32 × 32]
  ✓ Output  Out = Attn·V  [32 × 32]
  ✓ Q, K, V projections  [32 × 32] each (V: [32 × 32])
  ✓ Raw scores  S = Q·Kᵀ  [32 × 32]
  ✓ Scaled scores  (÷ √32 = 0.1768)
  ✓ Softmax attention weights  [32 × 32]
  ✓ Output  Out = Attn·V  [32 × 32]
  ✓ Q, K, V projections  [32 × 32] each (V: [32 × 32])
  ✓ Raw scores  S = Q·Kᵀ  [32 × 32]
  ✓ Scaled scores  (÷ √32 = 0.1768)
  ✓ Softmax attention weights  [32 × 32]
  ✓ Output  Out = Attn·V  [32 × 32]
  ✓ Q, K, V projections  [32 × 32] each (V: [32 × 32])
  ✓ Raw scores 

## 7. Visualise Attention Weights

In [27]:
import traceback, math
from IPython.display import display
try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker

    def get_attention_weights(X, Wq, Wk):
        Q = X @ Wq;  K = X @ Wk
        d_k = Q.shape[-1]
        S = Q @ K.T / math.sqrt(d_k)
        S -= S.max(axis=1, keepdims=True)
        A = np.exp(S);  A /= A.sum(axis=1, keepdims=True)
        return A

    SEQ_VIZ, D_MODEL_V, D_K_V = 32, 64, 32
    rng_v = np.random.default_rng(7)
    Xv  = rng_v.standard_normal((SEQ_VIZ, D_MODEL_V)).astype(np.float32)
    Wqv = rng_v.standard_normal((D_MODEL_V, D_K_V)).astype(np.float32) * 0.02
    Wkv = rng_v.standard_normal((D_MODEL_V, D_K_V)).astype(np.float32) * 0.02
    Wvv = rng_v.standard_normal((D_MODEL_V, D_K_V)).astype(np.float32) * 0.02

    attn_np    = get_attention_weights(Xv, Wqv, Wkv)
    cuda_out_v = cuda_self_attention(Xv, Wqv, Wkv, Wvv)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    im = ax.imshow(attn_np, aspect='auto', cmap='viridis', vmin=0)
    ax.set_title(f'Attention Weights  ({SEQ_VIZ}×{SEQ_VIZ})', fontsize=13)
    ax.set_xlabel('Key position');  ax.set_ylabel('Query position')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(4))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(4))

    ax2 = axes[1]
    row_sums = attn_np.sum(axis=1)
    ax2.plot(row_sums, 'o-', markersize=4, color='steelblue', label='row sum')
    ax2.axhline(1.0, color='tomato', linestyle='--', linewidth=1.5, label='expected = 1')
    ax2.set_title('Softmax Row-Sum Sanity Check', fontsize=13)
    ax2.set_xlabel('Query position');  ax2.set_ylabel('Sum of attention weights')
    ax2.set_ylim(0.95, 1.05);  ax2.legend();  ax2.grid(True, alpha=0.3)

    plt.suptitle('CUDA Self-Attention — Attention Weight Visualisation', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('attention_weights.png', dpi=120, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    print(f"Max deviation from row-sum=1: {abs(row_sums - 1).max():.2e}")
    print("Figure saved → attention_weights.png")
except Exception:
    traceback.print_exc()

  ✓ Q, K, V projections  [32 × 32] each (V: [32 × 32])
  ✓ Raw scores  S = Q·Kᵀ  [32 × 32]
  ✓ Scaled scores  (÷ √32 = 0.1768)
  ✓ Softmax attention weights  [32 × 32]
  ✓ Output  Out = Attn·V  [32 × 32]


<Figure size 1400x500 with 3 Axes>

Max deviation from row-sum=1: 5.96e-08
Figure saved → attention_weights.png


In [28]:
import time, io, contextlib

# ── GPU-only self-attention: data already lives on device ─────────────────────
def cuda_self_attention_device(X_gpu, Wq_gpu, Wk_gpu, Wv_gpu,
                                seq_len, d_model, d_k, d_v):
    """Same pipeline as cuda_self_attention but inputs/outputs stay on GPU.
    No H↔D transfers — isolates pure compute time."""
    def proj(X_gpu, W_gpu, d_out):
        out_gpu, _ = gpu_zeros((seq_len, d_out))
        grid  = (ceil_div(d_out, TILE), ceil_div(seq_len, TILE), 1)
        block = (TILE, TILE, 1)
        linproj_fn(X_gpu, W_gpu, out_gpu,
                   np.int32(seq_len), np.int32(d_model), np.int32(d_out),
                   block=block, grid=grid)
        return out_gpu

    Q_gpu  = proj(X_gpu, Wq_gpu, d_k)
    K_gpu  = proj(X_gpu, Wk_gpu, d_k)
    V_gpu  = proj(X_gpu, Wv_gpu, d_v)
    Kt_gpu = transpose_gpu(K_gpu, rows=seq_len, cols=d_k)
    S_gpu  = matmul_gpu(Q_gpu, Kt_gpu, seq_len, d_k, seq_len)

    scale = np.float32(1.0 / math.sqrt(d_k))
    n_el  = seq_len * seq_len
    scale_fn(S_gpu, scale, np.int32(n_el),
             block=(256, 1, 1), grid=(ceil_div(n_el, 256), 1, 1))

    threads_sm   = min(128, seq_len)
    shared_bytes = (threads_sm + seq_len) * 4
    softmax_fn(S_gpu, np.int32(seq_len),
               block=(threads_sm, 1, 1), grid=(seq_len, 1, 1),
               shared=shared_bytes)

    Out_gpu = matmul_gpu(S_gpu, V_gpu, seq_len, seq_len, d_v)

    for buf in [Q_gpu, K_gpu, V_gpu, Kt_gpu, S_gpu, Out_gpu]:
        buf.free()

def benchmark_gpu_only(X_gpu, Wq_gpu, Wk_gpu, Wv_gpu,
                        seq_len, d_model, d_k, d_v, warmup=3, runs=10):
    """Time GPU compute only — no H↔D transfers."""
    for _ in range(warmup):
        cuda_self_attention_device(X_gpu, Wq_gpu, Wk_gpu, Wv_gpu,
                                    seq_len, d_model, d_k, d_v)
    cuda.Context.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        cuda_self_attention_device(X_gpu, Wq_gpu, Wk_gpu, Wv_gpu,
                                    seq_len, d_model, d_k, d_v)
    cuda.Context.synchronize()
    return (time.perf_counter() - t0) / runs * 1e3

def benchmark_numpy(fn, *args, warmup=3, runs=10):
    for _ in range(warmup):
        fn(*args)
    t0 = time.perf_counter()
    for _ in range(runs):
        fn(*args)
    return (time.perf_counter() - t0) / runs * 1e3

large_configs = [
    ( 512,   512,  256,  256),
    (1024,  1024,  512,  512),
    (2048,  1024,  512,  512),
    (2048,  2048, 1024, 1024),
    (4096,  2048, 1024, 1024),
    (4096,  4096, 2048, 2048),
    (8192,  4096, 2048, 2048),
]

print(f"{'seq':>6} {'d_model':>8} {'d_k':>6}  {'CUDA (ms)':>11} {'NumPy (ms)':>11} {'Speedup':>9}  note")
print("─" * 72)

for seq, dm, dk, dv in large_configs:
    rng3 = np.random.default_rng(1)
    X3   = rng3.standard_normal((seq, dm)).astype(np.float32)
    Wq3  = rng3.standard_normal((dm, dk)).astype(np.float32) * 0.02
    Wk3  = rng3.standard_normal((dm, dk)).astype(np.float32) * 0.02
    Wv3  = rng3.standard_normal((dm, dv)).astype(np.float32) * 0.02

    # Pre-upload once — data lives on GPU for the whole benchmark
    X_gpu   = gpu_alloc_like(X3)
    Wq_gpu  = gpu_alloc_like(Wq3)
    Wk_gpu  = gpu_alloc_like(Wk3)
    Wv_gpu  = gpu_alloc_like(Wv3)

    t_cuda  = benchmark_gpu_only(X_gpu, Wq_gpu, Wk_gpu, Wv_gpu,
                                  seq, dm, dk, dv)
    t_numpy = benchmark_numpy(numpy_self_attention, X3, Wq3, Wk3, Wv3)
    speedup = t_numpy / t_cuda
    marker  = "◀ GPU faster!" if speedup > 1 else "◀ GPU slower"

    print(f"{seq:>6} {dm:>8} {dk:>6}  {t_cuda:>10.2f}  {t_numpy:>10.2f}  {speedup:>8.2f}×  {marker}")

    for buf in [X_gpu, Wq_gpu, Wk_gpu, Wv_gpu]:
        buf.free()

print()
print("Note: CUDA times exclude H↔D transfers (data pre-loaded on device,")
print("      matching real inference/training pipeline behaviour).")

   seq  d_model    d_k    CUDA (ms)  NumPy (ms)   Speedup  note
────────────────────────────────────────────────────────────────────────
   512      512    256        0.75        1.95      2.61×  ◀ GPU faster!
  1024     1024    512        3.54        8.15      2.31×  ◀ GPU faster!
  2048     1024    512        8.58       26.54      3.09×  ◀ GPU faster!
  2048     2048   1024       18.51       53.16      2.87×  ◀ GPU faster!
  4096     2048   1024       49.06      153.16      3.12×  ◀ GPU faster!
  4096     4096   2048      126.19      359.51      2.85×  ◀ GPU faster!
  8192     4096   2048      374.54     1002.95      2.68×  ◀ GPU faster!

Note: CUDA times exclude H↔D transfers (data pre-loaded on device,
      matching real inference/training pipeline behaviour).


In [22]:
import time

def benchmark_silent(fn, *args, warmup=3, runs=10):
    """Like benchmark() but suppresses per-step prints."""
    import io, contextlib
    sink = io.StringIO()
    for _ in range(warmup):
        with contextlib.redirect_stdout(sink):
            fn(*args)
    cuda.Context.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        with contextlib.redirect_stdout(sink):
            fn(*args)
    cuda.Context.synchronize()
    return (time.perf_counter() - t0) / runs * 1e3   # ms per call

large_configs = [
    # (seq_len, d_model, d_k, d_v)
    ( 256,  512,  256,  256),
    ( 512,  512,  256,  256),
    ( 512, 1024,  512,  512),
    (1024, 1024,  512,  512),
    (1024, 2048, 1024, 1024),
    (2048, 2048, 1024, 1024),
]

print(f"{'seq':>6} {'d_model':>8} {'d_k':>6}  {'CUDA (ms)':>11} {'NumPy (ms)':>11} {'Speedup':>9}")
print("─" * 58)

for seq, dm, dk, dv in large_configs:
    rng3 = np.random.default_rng(1)
    X3  = rng3.standard_normal((seq, dm)).astype(np.float32)
    Wq3 = rng3.standard_normal((dm, dk)).astype(np.float32) * 0.02
    Wk3 = rng3.standard_normal((dm, dk)).astype(np.float32) * 0.02
    Wv3 = rng3.standard_normal((dm, dv)).astype(np.float32) * 0.02

    t_cuda  = benchmark_silent(cuda_self_attention, X3, Wq3, Wk3, Wv3)
    t_numpy = benchmark_silent(numpy_self_attention, X3, Wq3, Wk3, Wv3)
    speedup = t_numpy / t_cuda
    marker  = " ◀ GPU faster!" if speedup > 1 else ""

    print(f"{seq:>6} {dm:>8} {dk:>6}  {t_cuda:>10.2f}  {t_numpy:>10.2f}  {speedup:>8.2f}×{marker}")

   seq  d_model    d_k    CUDA (ms)  NumPy (ms)   Speedup
──────────────────────────────────────────────────────────
   256      512    256        0.90        0.52      0.57×
   512      512    256        1.35        1.19      0.88×
   512     1024    512        2.79        3.03      1.09× ◀ GPU faster!
  1024     1024    512        4.98        7.54      1.51× ◀ GPU faster!
  1024     2048   1024       13.46       18.68      1.39× ◀ GPU faster!
  2048     2048   1024       26.42       49.84      1.89× ◀ GPU faster!
